# Molecular simulation in Colab

Choose Runtime → Change runtime type → **L4 GPU** (or A100), then Runtime → Run all. The notebook selects GPU4PySCF automatically when an NVIDIA GPU is available and uses CPU on CPU runtimes. Set `COLABEXERCISE_DEVICE=gpu` to require GPU execution. CUDA packages install into an isolated Python environment, preserving the active notebook's packages. QE and browser downloads are opt-in. TPU is not supported. Download results or copy them to Drive before the runtime ends.

## Get the project
Use an existing checkout, upload the project to `/content/colabexercise`, or clone GitHub. `COLABEXERCISE_REF` can select a published branch; the default is `main`. Thread limits are set before importing numerical libraries.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[key] = '1'
runtime_root = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
project = Path.cwd() if (Path.cwd() / 'quantum_workflow/workflow.py').is_file() else runtime_root / 'colabexercise'
repo_url = 'https://github.com/gutama/colabexercise.git'
branch = os.environ.get('COLABEXERCISE_REF', 'main')
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(project)], check=True)
if not (project / 'pyproject.toml').is_file():
    raise RuntimeError('Project files are missing; upload the project or select its published branch.')
artifact_root = Path(os.environ.get('COLABEXERCISE_OUTPUT', str(runtime_root)))
artifact_root.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'Project:', project)

In [ ]:
sys.path.insert(0, str(project))
from quantum_workflow.colab_runtime import prepare_environment, resolve_device

DEVICE = os.environ.get('COLABEXERCISE_DEVICE', 'auto')  # 'gpu' requires a GPU; 'cpu' selects CPU.
device = resolve_device(DEVICE)
runtime_python = prepare_environment(project, device, runtime_root / 'colabexercise-gpu-env')
print('SCF device:', device, 'Python:', runtime_python)

## Verify the NVIDIA GPU
CUDA 12 and 13 toolkits select separate, pinned GPU4PySCF/CuPy/cuTENSOR packages. The check below executes a CUDA operation and verifies cuTENSOR loading. Explicit GPU requests fail if hardware or dependencies are unavailable. SCF/DFT runs on GPU; full molecular-integral export and the Li-hBN NumPy demo use CPU.
CUDA 13 requires compute capability 8.0 or newer (L4/A100). T4 requires a CUDA 12 runtime.

In [ ]:
if device == 'gpu':
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.colab_runtime'], check=True)
else:
    print('CPU runtime selected. Choose a GPU runtime to enable GPU4PySCF.')

## PySCF: H₂ and integral export
GPU mode uses density fitting. For energy comparisons, use `density_fitting=true` on the CPU reference too. Change the example JSON for other systems; `summary.json` records the actual solver, GPU model, CUDA/package versions, and integral conventions.

In [ ]:
import json

config_data = json.loads((project / 'examples/h2-pyscf.json').read_text())
config_data['device'] = device
pyscf_output = Path(tempfile.mkdtemp(prefix='pyscf-', dir=artifact_root))
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', dir=runtime_root, delete=False) as config_file:
    json.dump(config_data, config_file)
    config_path = Path(config_file.name)
try:
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.cli', str(config_path),
                    '--output', str(pyscf_output)], check=True)
finally:
    config_path.unlink()
summary = json.loads((pyscf_output / 'summary.json').read_text())
print(summary)
print('Output:', pyscf_output)

## Optional QE calculation (CPU)
Downloads micromamba and installs a separate conda-forge QE 7.6 CPU environment, plus an example H PBE pseudopotential. This takes several minutes and disk space. QE uses a periodic supercell; independently test vacuum and cutoffs. Review downloaded software and pseudopotential suitability and provenance before scientific use. This section does not export VQE-ready integrals.

In [ ]:
RUN_QE = os.environ.get('COLABEXERCISE_RUN_QE', '0') == '1'  # Set True to opt in.
if RUN_QE:
    import tarfile
    import shutil

    mamba = runtime_root / 'micromamba'
    qe_env = runtime_root / 'qe-env'
    if not (qe_env / 'bin/pw.x').exists():
        subprocess.run(['curl', '-fL', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', str(runtime_root / 'micromamba.tar.bz2')], check=True)
        with tarfile.open(str(runtime_root / 'micromamba.tar.bz2'), 'r:bz2') as tar:
            source = tar.extractfile('bin/micromamba')
            if source is None:
                raise RuntimeError('micromamba archive is missing its executable')
            with source, mamba.open('wb') as target:
                shutil.copyfileobj(source, target)
        mamba.chmod(0o755)
        subprocess.run([str(mamba), 'create', '-y', '-r', str(runtime_root / 'mamba-root'), '-p', str(qe_env), '-c', 'conda-forge', '--override-channels', 'qe=7.6'], check=True)
    os.environ['PATH'] = str(qe_env / 'bin') + os.pathsep + os.environ['PATH']
    pseudo_dir = project / 'pseudopotentials'
    pseudo_dir.mkdir(exist_ok=True)
    pseudo = pseudo_dir / 'H.pbe-kjpaw_psl.1.0.0.UPF'
    subprocess.run(['curl', '-fL', 'https://pseudopotentials.quantum-espresso.org/upf_files/H.pbe-kjpaw_psl.1.0.0.UPF', '-o', str(pseudo)], check=True)
    qe_output = Path(tempfile.mkdtemp(prefix='qe-', dir=artifact_root))
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.cli', str(project / 'examples/h2-qe.json'),
                    '--output', str(qe_output)], check=True)
    print(json.loads((qe_output / 'summary.json').read_text()))
    print('Output:', qe_output)
else:
    print('QE skipped. Set RUN_QE = True to install QE and run the CPU example.')

## Download PySCF results (opt-in)
The archive includes integrals and metadata. Set `DOWNLOAD_RESULTS=True` to open a Colab browser download. For QE, archive `qe_output` similarly; its scratch directory can be large. See the README for tensor conventions and an FCI consumption example.

In [ ]:
DOWNLOAD_RESULTS = False  # True for a browser download in Google Colab.
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    archive = shutil.make_archive(str(artifact_root / 'pyscf-results'), 'zip', pyscf_output)
    files.download(archive)
else:
    print('Results saved:', pyscf_output)